In [21]:
import numpy as np 
import pandas as pd

In [22]:
from pathlib import Path
import json

import pandas as pd
from PIL import Image, UnidentifiedImageError

Data_DIr = Path(r"E:\Shortcutscope\archive\waterbird_complete95_forest2water2")
Metadata_Dir = Data_DIr / "metadata.csv"



In [23]:
df = pd.read_csv(Metadata_Dir)
df.shape

(11788, 6)

In [24]:
print(df.isnull().sum())

img_id            0
img_filename      0
y                 0
split             0
place             0
place_filename    0
dtype: int64


In [25]:
df.duplicated().sum()

np.int64(0)

In [26]:
required = {"img_filename", "y", "place", "split"}
if not required.issubset(df.columns):
    raise ValueError(f"Missing required columns: {required - set(df.columns)}")

In [27]:
for col in ["y", "place", "split"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

valid = (
    df["img_filename"].notna()
    & df["y"].isin([0, 1])
    & df["place"].isin([0, 1])
    & df["split"].isin([0, 1, 2])
)

In [28]:
def image_is_valid(image_path):
    filename = image_path
    if pd.isna(filename):
        return False
    path = (Data_DIr / str(filename)).resolve()
    if not path.is_relative_to(Data_DIr):
        return False
    if not path.is_file():
        return False
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except (OSError, ValueError):
        return False

In [30]:
df["image_valid"] = df["img_filename"].apply(image_is_valid)


In [32]:
print("\nInvalid metadata rows:", (~valid).sum())
print("Missing/unreadable image rows:", (~df["image_valid"]).sum())


Invalid metadata rows: 0
Missing/unreadable image rows: 0


In [33]:
cleaned = df.loc[valid & df["image_valid"]].copy()

# Remove exact duplicate metadata rows.
cleaned = cleaned.drop(
    columns=["image_valid"],
    errors="ignore"
)
cleaned = cleaned.drop_duplicates(keep="first")

# Restore integer label types
for col in ["y", "place", "split"]:
    cleaned[col] = cleaned[col].astype(int)

print("\nCleaned dataset shape:", cleaned.shape)
print("Rows removed:", len(df) - len(cleaned))

print("\nCleaned split counts:")
print(cleaned["split"].value_counts().sort_index())

print("\nCleaned group counts:")
print(
    pd.crosstab(
        2 * cleaned["y"] + cleaned["place"],
        cleaned["split"]
    )
)


Cleaned dataset shape: (11788, 6)
Rows removed: 0

Cleaned split counts:
split
0    4795
1    1199
2    5794
Name: count, dtype: int64

Cleaned group counts:
split     0    1     2
row_0                 
0      3498  467  2255
1       184  466  2255
2        56  133   642
3      1057  133   642
